# BPR (Bayesian Personalized Ranking) Experiments on MovieLens 25M

**Objective:** So sánh BPR vs ALS trên cùng dataset/split/protocol

**Protocol:** Sampled ranking (1 positive + 99 negatives per user)

**Metrics:** HitRate@K, NDCG@K (K=5,10,20)

## 1. Setup & Import Dependencies

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

# Add project root to path
_current = Path.cwd().resolve()
PROJECT_ROOT = _current.parent if _current.name == "notebooks" else _current
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import modules
from src.data_loader import ML25MLoader, DataPreprocessor
from src.recommenders import BPRRecommender, ALSRecommender

# Config
RANDOM_SEED = 42
POSITIVE_THRESHOLD = 4.0
np.random.seed(RANDOM_SEED)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✓ Import thành công tất cả thư viện")

## 2. Helper Functions

### 2.1 Temporal Leave-One-Positive-Out Split

In [ ]:
def temporal_leave_one_positive_out(
    ratings_df: pd.DataFrame, 
    threshold: float = 4.0
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hold out each user's final positive rating for BPR evaluation.
    
    Protocol:
    - Sort by (userId, timestamp)
    - Find last positive rating (rating >= threshold) per user
    - Require >=2 positive ratings in history
    - Split: everything before last positive → train, last positive → test
    
    Args:
        ratings_df: DataFrame with columns [userId, movieId, rating, timestamp]
        threshold: Minimum rating to consider as positive interaction
        
    Returns:
        train_bpr: All positive ratings before last positive per user
        test_positive: One positive test item per user
    """
    print(f"Original ratings: {len(ratings_df):,}")
    
    # Sort by user and time
    sorted_df = ratings_df.sort_values(["userId", "timestamp"]).reset_index(drop=True)
    
    train_parts = []
    test_rows = []
    
    for user_id, user_ratings in sorted_df.groupby("userId", sort=False):
        # Find positive ratings
        positive_mask = user_ratings["rating"] >= threshold
        positive_indices = np.flatnonzero(positive_mask)
        
        # Need at least 2 positives (1 for train, 1 for test)
        if len(positive_indices) < 2:
            continue
        
        # Last positive index in this user's group
        test_idx = positive_indices[-1]
        
        # Everything before test item
        history = user_ratings.iloc[:test_idx].copy()
        
        # Check if history has at least 1 positive
        if (history["rating"] >= threshold).sum() == 0:
            continue
        
        # Only keep positive ratings in train
        train_parts.append(history[history["rating"] >= threshold])
        test_rows.append(user_ratings.iloc[test_idx])
    
    train_bpr = pd.concat(train_parts, ignore_index=True)
    test_positive = pd.DataFrame(test_rows).reset_index(drop=True)
    
    print(f"Train BPR: {len(train_bpr):,} positive interactions")
    print(f"Test positive: {len(test_positive):,} users")
    print(f"Unique users in train: {train_bpr['userId'].nunique():,}")
    print(f"Unique movies in train: {train_bpr['movieId'].nunique():,}")
    
    return train_bpr, test_positive

print("✓ Defined temporal_leave_one_positive_out()")

### 2.2 Build Ranking Candidates

In [ ]:
def build_ranking_candidates(
    train_positive: pd.DataFrame,
    test_positive: pd.DataFrame,
    all_movie_ids: np.ndarray,
    negatives_per_user: int = 99,
    random_seed: int = 42,
) -> dict[int, np.ndarray]:
    """
    Create sampled ranking candidates: 1 positive + N negatives per user.
    
    Protocol:
    - For each test user, take their held-out positive item
    - Sample N unseen items (not in train) as negatives
    - Return candidates array: [positive_item, negative_1, ..., negative_N]
    
    Args:
        train_positive: Training interactions
        test_positive: Test interactions (one per user)
        all_movie_ids: Full catalogue of movie IDs
        negatives_per_user: Number of negative samples (default: 99)
        random_seed: Random seed for reproducibility
        
    Returns:
        Dictionary: {user_id: candidates_array}
    """
    rng = np.random.default_rng(random_seed)
    
    # Build seen items per user from training data
    seen = train_positive.groupby("userId")["movieId"].agg(set).to_dict()
    
    # Full catalogue
    catalog = set(map(int, all_movie_ids))
    
    candidates = {}
    
    for row in test_positive.itertuples(index=False):
        user_id = int(row.userId)
        positive_item = int(row.movieId)
        
        # Items to exclude: seen in train + the test positive item
        excluded = seen[user_id] | {positive_item}
        
        # Pool of potential negatives
        pool = np.fromiter(catalog - excluded, dtype=np.int32)
        
        # Sample negatives (or take all if fewer than requested)
        sample_size = min(negatives_per_user, len(pool))
        negatives = rng.choice(pool, size=sample_size, replace=False)
        
        # Candidates: positive item first, then negatives
        candidates[user_id] = np.concatenate(([positive_item], negatives))
    
    print(f"Built ranking candidates for {len(candidates):,} users")
    print(f"Candidates per user: 1 positive + {negatives_per_user} negatives")
    
    return candidates

print("✓ Defined build_ranking_candidates()")

### 2.3 Evaluation Function

In [ ]:
def evaluate_ranking_at_k(
    model,
    candidate_map: dict[int, np.ndarray],
    k: int = 10,
) -> dict[str, float]:
    """
    Evaluate model with HitRate@K and NDCG@K using sampled ranking protocol.
    
    Protocol:
    - For each user, rank their candidate set (1 positive + N negatives)
    - Find rank of the positive item (candidates[0])
    - HitRate@K: 1 if positive in top-K, else 0
    - NDCG@K: DCG with position discount if positive in top-K, else 0
    
    Args:
        model: Trained BPR or ALS recommender with .recommend() method
        candidate_map: {user_id: candidates_array} from build_ranking_candidates()
        k: Cutoff for evaluation metrics
        
    Returns:
        Dictionary with hit_rate, ndcg, and users count
    """
    hits = []
    ndcgs = []
    
    for user_id, candidates in candidate_map.items():
        # Get recommendations from model (returns list of movieIds)
        try:
            recommended = model.recommend(
                user_id=user_id,
                n=len(candidates),
                seen=None  # Model already filters train items
            )
        except Exception as e:
            recommended = []
        
        if not recommended:
            hits.append(0.0)
            ndcgs.append(0.0)
            continue
        
        # Find rank of positive item (candidates[0])
        positive_item = candidates[0]
        
        # Filter recommendations to only those in candidate set
        candidate_set = set(candidates)
        filtered_recs = [mid for mid in recommended if mid in candidate_set]
        
        # Find position
        try:
            rank = filtered_recs.index(positive_item)
        except ValueError:
            rank = len(filtered_recs)  # Not found
        
        # Compute metrics
        if rank < k:
            hits.append(1.0)
            # NDCG discount: 1 / log2(rank + 2)
            ndcgs.append(1.0 / np.log2(rank + 2))
        else:
            hits.append(0.0)
            ndcgs.append(0.0)
    
    return {
        "hit_rate": float(np.mean(hits)),
        "ndcg": float(np.mean(ndcgs)),
        "users": len(hits),
    }

print("✓ Defined evaluate_ranking_at_k()")

## 3. Load Data

In [ ]:
# Load MovieLens 25M data
# Use sample_frac=0.1 for quick testing
loader = ML25MLoader()
print("Loading ratings... (this may take a few minutes)")

# OPTION 1: Full dataset (uncomment for production run)
# ratings = loader.load_ratings()

# OPTION 2: 10% sample for testing (faster)
ratings = loader.load_ratings(sample_frac=0.1)

print(f"Loaded {len(ratings):,} ratings")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['movieId'].nunique():,}")
print(f"Timespan: {pd.to_datetime(ratings['timestamp'], unit='s').min()} to {pd.to_datetime(ratings['timestamp'], unit='s').max()}")

## 4. Data Split

In [ ]:
# Apply temporal leave-one-positive-out split
train_bpr, test_positive = temporal_leave_one_positive_out(
    ratings, threshold=POSITIVE_THRESHOLD
)

# Filter cold-start: ensure test items are warm
warm_users = set(train_bpr["userId"].unique())
warm_movies = set(train_bpr["movieId"].unique())

test_positive = test_positive[
    test_positive["userId"].isin(warm_users) & 
    test_positive["movieId"].isin(warm_movies)
].copy()

print(f"\nAfter cold-start filtering:")
print(f"Test positive (warm): {len(test_positive):,} users")

# Save intermediate results
output_dir = PROJECT_ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
train_bpr.to_csv(output_dir / "train_bpr_ml25m.csv", index=False)
test_positive.to_csv(output_dir / "test_positive_ml25m.csv", index=False)
print(f"✓ Saved to {output_dir}/")

## 5. Train BPR Model

In [ ]:
# Initialize BPR model
print("Training BPR model...")
print("Configuration:")
print("  - Factors: 64")
print("  - Learning rate: 0.05")
print("  - Regularization: 0.01")
print("  - Iterations: 100")

start_time = time.perf_counter()

bpr = BPRRecommender(
    factors=64,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    verify_negative_samples=True,
    random_state=RANDOM_SEED
)

bpr.fit(train_bpr)

bpr_train_time = time.perf_counter() - start_time
print(f"\n✓ BPR training completed in {bpr_train_time:.2f} seconds ({bpr_train_time/60:.2f} minutes)")

# Quick sanity check
sample_user_id = int(train_bpr['userId'].iloc[0])
sample_recs = bpr.recommend(user_id=sample_user_id, n=10)
print(f"\nSanity check - Top 10 recommendations for user {sample_user_id}:")
print(f"  {sample_recs[:5]}... (showing first 5)")

## 6. Train ALS Model (for comparison)

In [ ]:
# Initialize ALS model with same number of factors as BPR
print("Training ALS model...")
print("Configuration:")
print("  - Factors: 64")
print("  - Regularization: 0.01")
print("  - Iterations: 15")
print("  - Alpha: 1.0")

start_time = time.perf_counter()

als = ALSRecommender(
    factors=64,
    regularization=0.01,
    iterations=15,
    alpha=1.0,
    random_state=RANDOM_SEED
)

als.fit(train_bpr)

als_train_time = time.perf_counter() - start_time
print(f"\n✓ ALS training completed in {als_train_time:.2f} seconds ({als_train_time/60:.2f} minutes)")

# Training time comparison
print(f"\nTraining Time Comparison:")
print(f"  BPR: {bpr_train_time:.2f}s ({bpr.iterations} iterations)")
print(f"  ALS: {als_train_time:.2f}s ({als.iterations} iterations)")
print(f"  Ratio (BPR/ALS): {bpr_train_time / als_train_time:.2f}x")

# Sanity check for ALS
als_recs = als.recommend(user_id=sample_user_id, n=10)
print(f"\nSanity check - Top 10 recommendations for user {sample_user_id} (ALS):")
print(f"  {als_recs[:5]}... (showing first 5)")

## 7. Build Ranking Candidates

In [ ]:
# Build ranking candidates for evaluation
item_ids = np.sort(train_bpr["movieId"].unique())

ranking_candidates = build_ranking_candidates(
    train_bpr, 
    test_positive, 
    item_ids, 
    negatives_per_user=99,
    random_seed=RANDOM_SEED
)

print(f"\n✓ Ranking candidates prepared")
sample_user = list(ranking_candidates.keys())[0]
print(f"Sample candidate set for user {sample_user}:")
print(f"  Positive: {ranking_candidates[sample_user][0]}")
print(f"  Negatives (first 5): {ranking_candidates[sample_user][1:6]}")

## 8. Evaluation

In [ ]:
# Evaluate BPR at different K values
print("\nEvaluating BPR model...")
bpr_results = {}

for k in [5, 10, 20]:
    result = evaluate_ranking_at_k(bpr, ranking_candidates, k=k)
    bpr_results[k] = result
    print(f"BPR @ K={k}:")
    print(f"  HitRate@{k}: {result['hit_rate']:.4f}")
    print(f"  NDCG@{k}: {result['ndcg']:.4f}")
    print(f"  Users evaluated: {result['users']:,}")

In [ ]:
# Evaluate ALS with SAME protocol as BPR for fair comparison
print("\nEvaluating ALS model with same protocol...")
als_results = {}

for k in [5, 10, 20]:
    result = evaluate_ranking_at_k(als, ranking_candidates, k=k)
    als_results[k] = result
    print(f"ALS @ K={k}:")
    print(f"  HitRate@{k}: {result['hit_rate']:.4f}")
    print(f"  NDCG@{k}: {result['ndcg']:.4f}")
    print(f"  Users evaluated: {result['users']:,}")

## 9. Results Summary

In [ ]:
# Summary comparison
print("\n" + "="*70)
print("SUMMARY: BPR vs ALS on MovieLens 25M (Sampled Ranking Protocol)")
print("="*70)

for k in [5, 10, 20]:
    print(f"\n@ K={k}:")
    print(f"  BPR - HitRate: {bpr_results[k]['hit_rate']:.4f}, NDCG: {bpr_results[k]['ndcg']:.4f}")
    print(f"  ALS - HitRate: {als_results[k]['hit_rate']:.4f}, NDCG: {als_results[k]['ndcg']:.4f}")
    
    hr_diff = bpr_results[k]['hit_rate'] - als_results[k]['hit_rate']
    ndcg_diff = bpr_results[k]['ndcg'] - als_results[k]['ndcg']
    
    better_hr = "BPR" if hr_diff > 0 else "ALS"
    better_ndcg = "BPR" if ndcg_diff > 0 else "ALS"
    
    print(f"  Better HitRate: {better_hr} ({abs(hr_diff):.4f} difference)")
    print(f"  Better NDCG: {better_ndcg} ({abs(ndcg_diff):.4f} difference)")

print(f"\nTraining Time:")
print(f"  BPR: {bpr_train_time:.2f}s")
print(f"  ALS: {als_train_time:.2f}s")
print("="*70)

## 10. Visualization

In [ ]:
# Create comparison DataFrame
comparison_data = []
for k in [5, 10, 20]:
    comparison_data.append({
        "K": k,
        "BPR_HitRate": bpr_results[k]["hit_rate"],
        "BPR_NDCG": bpr_results[k]["ndcg"],
        "ALS_HitRate": als_results[k]["hit_rate"],
        "ALS_NDCG": als_results[k]["ndcg"],
    })

df_results = pd.DataFrame(comparison_data)

print("\nComparison Table:")
print(df_results.to_string(index=False))

# Save results to CSV
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
df_results.to_csv(results_dir / "bpr_vs_als_comparison.csv", index=False)
print(f"\n✓ Saved comparison table to {results_dir / 'bpr_vs_als_comparison.csv'}")

In [ ]:
# Visualization: Bar charts comparing BPR vs ALS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HitRate comparison
k_values = [5, 10, 20]
bpr_hitrates = [bpr_results[k]["hit_rate"] for k in k_values]
als_hitrates = [als_results[k]["hit_rate"] for k in k_values]

x = np.arange(len(k_values))
width = 0.35

axes[0].bar(x - width/2, bpr_hitrates, width, label='BPR', color='#2ecc71', alpha=0.8)
axes[0].bar(x + width/2, als_hitrates, width, label='ALS', color='#3498db', alpha=0.8)
axes[0].set_xlabel('K', fontsize=12)
axes[0].set_ylabel('HitRate@K', fontsize=12)
axes[0].set_title('HitRate@K: BPR vs ALS', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(k_values)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# NDCG comparison
bpr_ndcgs = [bpr_results[k]["ndcg"] for k in k_values]
als_ndcgs = [als_results[k]["ndcg"] for k in k_values]

axes[1].bar(x - width/2, bpr_ndcgs, width, label='BPR', color='#2ecc71', alpha=0.8)
axes[1].bar(x + width/2, als_ndcgs, width, label='ALS', color='#3498db', alpha=0.8)
axes[1].set_xlabel('K', fontsize=12)
axes[1].set_ylabel('NDCG@K', fontsize=12)
axes[1].set_title('NDCG@K: BPR vs ALS', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(k_values)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()

# Save figure
figures_dir = PROJECT_ROOT / "reports/figures"
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / 'bpr_vs_als_comparison.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved visualization to {figures_dir / 'bpr_vs_als_comparison.png'}")

plt.show()

## 11. Export Models & Results

In [ ]:
# Save structured results to JSON for documentation
results_json = {
    "dataset": "MovieLens 25M",
    "sample_fraction": 0.1,
    "split": "temporal_leave_one_positive_out",
    "protocol": "sampled_ranking_1_plus_99",
    "positive_threshold": POSITIVE_THRESHOLD,
    "seed": RANDOM_SEED,
    "train_size": len(train_bpr),
    "test_size": len(test_positive),
    "bpr": {
        "config": {
            "factors": bpr.factors,
            "learning_rate": bpr.learning_rate,
            "regularization": bpr.regularization,
            "iterations": bpr.iterations,
        },
        "train_time_seconds": bpr_train_time,
        "results": {
            f"k{k}": {
                "hit_rate": bpr_results[k]["hit_rate"],
                "ndcg": bpr_results[k]["ndcg"],
                "users": bpr_results[k]["users"],
            }
            for k in [5, 10, 20]
        }
    },
    "als": {
        "config": {
            "factors": als.factors,
            "regularization": als.regularization,
            "iterations": als.iterations,
            "alpha": als.alpha,
        },
        "train_time_seconds": als_train_time,
        "results": {
            f"k{k}": {
                "hit_rate": als_results[k]["hit_rate"],
                "ndcg": als_results[k]["ndcg"],
                "users": als_results[k]["users"],
            }
            for k in [5, 10, 20]
        }
    }
}

import json as json_module
with open(results_dir / "bpr_ml25m_results.json", "w") as f:
    json_module.dump(results_json, f, indent=2)

print(f"✓ Saved structured results to {results_dir / 'bpr_ml25m_results.json'}")

In [ ]:
# Save best models
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(exist_ok=True)

# Save BPR model
bpr.save(str(models_dir / "bpr_best.pkl"))
print(f"✓ Saved BPR model to {models_dir / 'bpr_best.pkl'}")

# Save ALS model for comparison
als.save(str(models_dir / "als_bpr_comparison.pkl"))
print(f"✓ Saved ALS model to {models_dir / 'als_bpr_comparison.pkl'}")

print("\n" + "="*70)
print("All results and models saved successfully!")
print("="*70)

## 12. Conclusion

### Key Findings

1. **Performance Comparison**: [Kết quả sẽ được cập nhật sau khi chạy notebook]
2. **Training Time**: BPR thường chậm hơn ALS do nhiều iterations hơn
3. **Use Cases**:
   - BPR: Tối ưu cho Top-N ranking tasks
   - ALS: Cân bằng giữa performance và training time

### Next Steps

1. Hyperparameter tuning cho BPR (learning rate, iterations)
2. Thử nghiệm trên full dataset (không sample)
3. So sánh với các metrics khác (MAP, MRR)
4. Ensemble BPR + ALS cho recommendations